In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch
import os
import sys

%matplotlib inline

if "snakemake" not in globals():
    from _helpers_notebooks import mock_snakemake
    snakemake = mock_snakemake(
        "plot_mga",
        wacc="regional",
        trade_chain="supplyconstraint",
    )

# MGA Analysis: HBI Production by Political Stability

### Load snakemake and configuration

In [ ]:
import os
import re
import glob
import pandas as pd
import pypsa
from pypsa import Network

def extract_scenario(path):
    """Extract scenario name from a results path like '.../scenario~mga-stability-weighted/...'"""
    match = re.search(r"scenario~([^/\\]+)", path)
    if match:
        return match.group(1)
    raise ValueError(f"Could not extract scenario name from path: {path}")

# Derive scenario names directly from the Snakemake input paths
scenario            = extract_scenario(snakemake.input.network_mga_production)
cp_scenario         = extract_scenario(snakemake.input.network_mga_chokepoints)
bl_scenario         = extract_scenario(snakemake.input.network_mga_blocks)

# Access MGA scenario configuration from snakemake.config
mga_config = snakemake.config.get("scenario", {}).get(scenario, {})
mga_modifiers = mga_config.get("modifiers", {})
mga_slack_values = mga_modifiers.get("mga", {}).get("slack", [])
cost_year = "2050"
interone = "hbi"
intertwo = "eaf-grid"
final = "steel"
region_nice_names = snakemake.config.get("region_nice_names", {})

print(f"Production scenario:  {scenario}")
print(f"Chokepoint scenario:  {cp_scenario}")
print(f"Blocks scenario:      {bl_scenario}")
print(f"\nMGA Configuration: {mga_config}")
print(f"MGA Modifiers: {mga_modifiers}")
print(f"Slack values from config: {mga_slack_values}")


### Load PyPSA networks using NetworkCollection

In [ ]:
# Construct path to results directory
results_dir = os.path.abspath(os.path.join(
    os.path.dirname(snakemake.input.network_mga_production),
    "..",
    scenario
))

if not os.path.exists(results_dir):
    results_dir = os.path.abspath(os.path.join(
        os.path.dirname(snakemake.input.network_mga_production)
    ))

print(f"Results directory: {results_dir}")

# Load networks into a NetworkCollection
networks_dict = {}
slack_values = [0] + mga_slack_values

print(f"\nLoading networks for slack values: {slack_values}")

for slack in slack_values:
    if slack == 0:
        filepath = os.path.join(results_dir, "network_nan.nc")
        slack_key = None
    else:
        filepath = os.path.join(results_dir, f"network_{slack}.nc")
        slack_key = slack
    
    if os.path.exists(filepath):
        try:
            networks_dict[slack_key] = pypsa.Network(filepath)
            label = "optimal solution (slack=0)" if slack == 0 else f"slack={slack}"
            print(f"  ✓ Loaded {label}")
        except Exception as e:
            print(f"  ✗ Error loading {filepath}: {e}")
    else:
        print(f"  ✗ File not found: {filepath}")

# Create NetworkCollection from the loaded networks
# (Store as dict for now, can use NetworkCollection if available in this PyPSA version)
networks = networks_dict

print(f"\n✓ Successfully loaded {len(networks_dict)} networks")
print(f"✓ Network keys: {sorted(networks_dict.keys(), key=lambda x: (x is None, x))}")

### Prepare MGA criteria: Political Stability

In [ ]:
stability_file = snakemake.input.political_stability

print(f"Loading political stability from: {stability_file}")
stability_df = pd.read_csv(stability_file, index_col=0)
print("Political stability data:")
print(stability_df)

# Define threshold from MGA config
threshold_stability = mga_config.get("modifiers", {}).get("mga", {}).get("threshold_value", [])

# Separate stable and unstable regions
stable_regions = stability_df[stability_df.iloc[:, 0] >= threshold_stability].index.tolist()
unstable_regions = stability_df[stability_df.iloc[:, 0] < threshold_stability].index.tolist()

print(f"\nStable regions (stability >= {threshold_stability}): {stable_regions}")
print(f"Unstable regions (stability < {threshold_stability}): {unstable_regions}")

all_regions = stable_regions + unstable_regions
print(f"\nTotal regions: {len(all_regions)}")
for region in sorted(all_regions):
    is_stable = region in stable_regions
    print(f"  {region}: {'STABLE' if is_stable else 'UNSTABLE'}")

### Read PyPSA networks and extract HBI production data

In [ ]:
# Sort network keys: None (optimal) first, then MGA variants in ascending order
sorted_keys = sorted(networks_dict.keys(), key=lambda x: (x is not None, x))

# Create epsilon array for columns (0 for optimal, then slack values)
epsilon_values = [0 if k is None else k for k in sorted_keys]

# Initialize storage for regional HBI production
hbi_production_data = {region: [] for region in all_regions}

# Extract HBI production from each network
for net_key in sorted_keys:
    n = networks_dict[net_key]
    
    # Initialize regional totals for this network
    regional_totals = {region: 0 for region in all_regions}
    
    # Get HBI production from links (input side: bus0, p0)
    # Note: HBI links have ore as bus0 and HBI as bus1, with efficiency factor
    # The actual production flows through p0 (input side of the conversion)
    links_df = n.links
    
    for link_idx, link_row in links_df.iterrows():
        if link_row['carrier'] == 'hbi':
            bus1 = link_row['bus1']  # Output bus (e.g., "Europe_hbi")
            
            # Get the HBI production from this link
            # Use p0 (input ore amount) to track production
            if hasattr(n, 'links_t') and 'p0' in n.links_t:
                p_sum = n.links_t['p0'][link_idx].sum()
            else:
                p_sum = link_row.get('p_nom_opt', 0)
            
            if not pd.isna(p_sum) and p_sum > 0:
                # Determine which region this HBI output belongs to
                # Extract region from bus1 (e.g., "Europe_hbi" -> "Europe")
                found = False
                for region in all_regions:
                    if bus1.startswith(region):
                        regional_totals[region] += p_sum
                        found = True
                        break
            
            
    
    # Append regional totals for this network
    for region in all_regions:
        hbi_production_data[region].append(regional_totals[region])



# Create pandas DataFrame with regions as rows and slack values as columns
# Convert from tonnes to Mt (divide by 1 million)
df_hbi = pd.DataFrame(hbi_production_data, index=epsilon_values).T / 1e6

# Adjust from input (iron ore) to output (HBI)
df_hbi = df_hbi / snakemake.config["iron_ore"]["ore_to_steel_ratio"]

# Rename index to be more descriptive
df_hbi.index.name = 'Region'
df_hbi.columns.name = 'MGA Slack (ε)'

print("HBI Production by Region and MGA Slack (in Mt):")
df_hbi.round(1)

### Append MGA criteria (political stability) to the dataframe

In [ ]:
# Add political stability column
df_hbi['Political Stability'] = stability_df.iloc[:, 0]

# Add stability classification (Stable/Unstable)
df_hbi['Stability Class'] = df_hbi.index.map(
    lambda region: 'Stable' if region in stable_regions else 'Unstable'
)
df_hbi

### Plot HBI production data

In [ ]:

# Extract epsilon values and production data
epsilon_cols = df_hbi.columns[:-2]  # All columns except 'Political Stability' and 'Stability Class'
epsilon_vals = np.array([float(x) for x in epsilon_cols])

# Aggregate by stability class
total_stable_mt = df_hbi[df_hbi['Stability Class'] == 'Stable'][epsilon_cols].sum()
total_unstable_mt = df_hbi[df_hbi['Stability Class'] == 'Unstable'][epsilon_cols].sum()

# Convert to numpy arrays
total_stable_mt = np.array(total_stable_mt.values, dtype=float)
total_unstable_mt = np.array(total_unstable_mt.values, dtype=float)

# Get individual regional data
stable_production_mt = {region: np.array(df_hbi.loc[region, epsilon_cols].values, dtype=float)
                        for region in stable_regions if region in df_hbi.index}
unstable_production_mt = {region: np.array(df_hbi.loc[region, epsilon_cols].values, dtype=float)
                          for region in unstable_regions if region in df_hbi.index}

# Load region colors from snakemake config
region_colors = snakemake.config["colors"]

# Create single figure
fig, ax = plt.subplots(1, 1, figsize=(12, 7))

# Settings
GAP_FRAC = 0.04

# ── Plot global (aggregated) line for stable regions (thicker, more prominent) ────
stable_color = "#3B6D11"
ax.plot(epsilon_vals, total_stable_mt, color=stable_color, linewidth=3, marker='o', 
        markersize=8, label="Global (Stable)", zorder=20)

# ── Plot all individual regional lines ────────────────────────────────────────

# Stable regions (solid lines)
for region in stable_regions:
    if region in stable_production_mt:
        region_data = stable_production_mt[region]
        if region_data.sum() > 0:  # Only plot if region has any production
            color = region_colors.get(region, '#000000')
            ax.plot(epsilon_vals, region_data, color=color, linewidth=1.2, 
                   linestyle='-', alpha=0.6, label=region_nice_names.get(region, region.replace("_", " ")), zorder=5)

# Unstable regions (dashed lines)
for region in unstable_regions:
    if region in unstable_production_mt:
        region_data = unstable_production_mt[region]
        if region_data.sum() > 0:  # Only plot if region has any production
            color = region_colors.get(region, '#000000')
            ax.plot(epsilon_vals, region_data, color=color, linewidth=1.2, 
                   linestyle='--', alpha=0.7, label=region_nice_names.get(region, region.replace("_", " ")), zorder=5)

# ── Axes formatting ──────────────────────────────────────────────────────────
ax.set_xlim(min(epsilon_vals) - 0.002, max(epsilon_vals) + 0.01)
ax.set_ylim(0, max(total_stable_mt) * 1.15)
ax.set_xlabel("MGA slack (fraction, ε)", fontsize=12)
ax.set_ylabel("HBI production in Mt", fontsize=12)
ax.set_title("HBI Production by Exporter Stability (All 16 Regions)")
ax.grid(True, linestyle="--", linewidth=0.5, alpha=0.5)
ax.legend(loc="best", fontsize=7, framealpha=0.95, ncol=3, title="Regions")

plt.tight_layout()
plt.show()


### Statistics

In [ ]:

print("✓ Plot with all 16 regions created successfully!")
print(f"\nStable regions HBI production range: {total_stable_mt.min():.4f} to {total_stable_mt.max():.4f} Mt")
if total_stable_mt[0] > 0:
    print(f"  Change: {((total_stable_mt[-1] / total_stable_mt[0]) - 1) * 100:.1f}%")
print(f"\nUnstable regions HBI production range: {total_unstable_mt.min():.4f} to {total_unstable_mt.max():.4f} Mt")
if total_unstable_mt[0] > 0:
    print(f"  Change: {((total_unstable_mt[-1] / total_unstable_mt[0]) - 1) * 100:.1f}%")

---
# MGA Analysis: HBI Trade through Maritime Chokepoints

Analyse how much HBI shipping trade passes through each maritime chokepoint using the **mga-chokepoints** scenario networks. This scenario minimises trade through chokepoint-exposed routes.

### Load chokepoint data and build route-to-chokepoint mapping

In [ ]:
# ── Load MGA-chokepoints scenario configuration ──
# cp_scenario is derived from snakemake.input.network_mga_chokepoints in the config cell above
cp_mga_config = snakemake.config.get("scenario", {}).get(cp_scenario, {})
cp_mga_modifiers = cp_mga_config.get("modifiers", {})
cp_mga_slack_values = cp_mga_modifiers.get("mga", {}).get("slack", [])

# Chokepoints listed in the config
config_chokepoints = cp_mga_modifiers.get("mga", {}).get("threshold_value", [])

print(f"Chokepoint MGA scenario: {cp_scenario}")
print(f"Slack values: {cp_mga_slack_values}")
print(f"Chokepoints from config: {config_chokepoints}")


# ── Load PyPSA networks from mga-chokepoints results ──
cp_results_dir = os.path.abspath(os.path.join(
        os.path.dirname(snakemake.input.network_mga_chokepoints)
    ))

print(f"\nResults directory: {cp_results_dir}")

cp_networks_dict = {}
cp_slack_values = [0] + cp_mga_slack_values

for slack in cp_slack_values:
    if slack == 0:
        filepath = os.path.join(cp_results_dir, "network_nan.nc")
        slack_key = None
    else:
        filepath = os.path.join(cp_results_dir, f"network_{slack}.nc")
        slack_key = slack

    if os.path.exists(filepath):
        try:
            cp_networks_dict[slack_key] = pypsa.Network(filepath)
            label = "optimal (slack=0)" if slack == 0 else f"slack={slack}"
            print(f"  ✓ Loaded {label}")
        except Exception as e:
            print(f"  ✗ Error loading {filepath}: {e}")
    else:
        print(f"  ✗ File not found: {filepath}")

print(f"\n✓ Loaded {len(cp_networks_dict)} chokepoint MGA networks")

# ── Sort keys and build epsilon array ──
cp_sorted_keys = sorted(cp_networks_dict.keys(), key=lambda x: (x is not None, x))
cp_epsilon_values = [0 if k is None else k for k in cp_sorted_keys]

# ── Load trade options with chokepoints column ──
trade_options_cp = pd.read_csv(snakemake.input.trade_options_chokepoints, header=0)

# Build mapping: (region_from, region_to) -> set of chokepoints
route_chokepoints = {}
for _, row in trade_options_cp.iterrows():
    r_from = row["region_from"]
    r_to = row["region_to"]
    cp_str = row.get("chokepoints", "")
    if pd.notna(cp_str) and cp_str != "":
        cp_set = set(str(cp_str).split(";"))
    else:
        cp_set = set()
    route_chokepoints[(r_from, r_to)] = cp_set

# Show summary: how many routes pass through each chokepoint
from collections import Counter
cp_count = Counter()
for cp_set in route_chokepoints.values():
    for cp in cp_set:
        if cp in config_chokepoints:
            cp_count[cp] += 1

print(f"\nRoutes per chokepoint (config only):")
for cp in config_chokepoints:
    print(f"  {cp}: {cp_count.get(cp, 0)} routes")


### Extract HBI shipping trade through chokepoints per MGA slack

In [ ]:
# For each chokepoint MGA network, sum HBI shipping trade through each chokepoint
# A link contributes to a chokepoint if its route passes through it.
# p_nom_opt gives the capacity allocated to each shipping link.

carrier = "hbi"

# Initialize: {chokepoint: [trade_per_epsilon...]}
cp_trade_data = {cp: [] for cp in config_chokepoints}
cp_trade_total = []  # total trade through any config chokepoint

for net_key in cp_sorted_keys:
    n_cp = cp_networks_dict[net_key]

    # Per-chokepoint totals for this network
    cp_totals = {cp: 0.0 for cp in config_chokepoints}
    total_through_any = 0.0

    # Iterate shipping HBI links
    shipping_links = n_cp.links[n_cp.links.carrier == f"shipping_{carrier}"]

    for link_name, link_row in shipping_links.iterrows():
        # Extract region_from, region_to from link name: "shipping hbi {r_from}-{r_to}"
        route_part = link_name.replace(f"shipping {carrier} ", "")
        parts = route_part.split("-")
        if len(parts) != 2:
            continue
        r_from, r_to = parts

        # Get trade volume (p_nom_opt = capacity in t)
        trade_vol = link_row.get("p_nom_opt", 0)
        if pd.isna(trade_vol) or trade_vol <= 0:
            continue

        # Check which config chokepoints this route passes through
        route_cp = route_chokepoints.get((r_from, r_to), set())
        matched_cp = route_cp & set(config_chokepoints)

        if matched_cp:
            total_through_any += trade_vol
            for cp in matched_cp:
                cp_totals[cp] += trade_vol

    cp_trade_total.append(total_through_any)
    for cp in config_chokepoints:
        cp_trade_data[cp].append(cp_totals[cp])

# Build DataFrame: rows = chokepoints, columns = epsilon values
df_cp = pd.DataFrame(cp_trade_data, index=cp_epsilon_values).T / 1e6  # Convert to Mt
df_cp.index.name = "Chokepoint"
df_cp.columns.name = "MGA Slack (ε)"

# "TOTAL (any chokepoint)": a route is counted once even if it passes multiple chokepoints
total_any_series = pd.Series(cp_trade_total, index=cp_epsilon_values, name="TOTAL (any chokepoint)") / 1e6

# "TOTAL (all chokepoints)": sum of all per-chokepoint rows —
# a route through N chokepoints is counted N times, reflecting total chokepoint-crossing exposure
total_all_series = df_cp.sum(axis=0).rename("TOTAL (all chokepoints)")

print("HBI shipping trade through chokepoints (Mt):")
df_cp_display = pd.concat([df_cp, total_any_series.to_frame().T, total_all_series.to_frame().T])
df_cp_display.round(1)


### Plot: HBI trade through chokepoints vs MGA slack

In [ ]:
epsilon_arr = np.array(cp_epsilon_values, dtype=float)

# Filter to chokepoints that actually have non-zero trade
active_cp = [cp for cp in config_chokepoints if df_cp.loc[cp].sum() > 0]
print(f"Active chokepoints (with trade): {active_cp}")

# Chokepoint display names (override .title() defaults)
cp_display_names = {
    "ormuz":       "Hormuz",
    "babalmandab": "Bab el-Mandeb",
}

# Chokepoint colors — read from config (mapped to geographically relevant region colors)
_colors_cfg = snakemake.config["colors"]
cp_colors = {cp: _colors_cfg.get(cp, "#333333") for cp in config_chokepoints}

# Chokepoint line styles (solid/dashed to distinguish geographically proximate pairs)
cp_linestyles = {
    "babalmandab": "-",
    "suez":        "--",   # Red Sea area: dashed to distinguish from babalmandab
    "bosporus":    "-",
    "gibraltar":   "-",   # European straits: dashed
    "ormuz":       "-",
    "panama":      "-",
    "malacca":     "-",
    "sunda":       "--",   # SE Asia pair: dashed
    "south_africa":"-",
    "northwest":   "--",
    "chili":       "--",
}

fig, ax = plt.subplots(1, 1, figsize=(12, 7))

# ── Plot "all chokepoints" total as thick black line ──
# (a route through N chokepoints is counted N times)
total_all_arr = np.array(df_cp_display.loc["TOTAL (all chokepoints)"].values, dtype=float)
ax.plot(epsilon_arr, total_all_arr, color="black", linewidth=3, marker="o",
        markersize=8, label="Total (all chokepoints, summed)", zorder=20)

# ── Also plot "any chokepoint" total as thin dashed line for reference ──
total_any_arr = np.array(cp_trade_total, dtype=float) / 1e6
ax.plot(epsilon_arr, total_any_arr, color="black", linewidth=1.5, marker="o",
        markersize=5, linestyle="--", alpha=0.5,
        label="Total (any chokepoint, deduplicated)", zorder=15)

# ── Plot individual chokepoints (disaggregated) ──
for cp in active_cp:
    cp_arr = np.array(df_cp.loc[cp].values, dtype=float)
    color = cp_colors.get(cp, "#333333")
    label = cp_display_names.get(cp, cp.replace("_", " ").title())
    ax.plot(epsilon_arr, cp_arr, color=color, linewidth=1.5,
            linestyle=cp_linestyles.get(cp, "-"), marker="s", markersize=4, alpha=0.8,
            label=label, zorder=10)

# ── Axes formatting ──
ax.set_xlim(min(epsilon_arr) - 0.002, max(epsilon_arr) + 0.01)
ax.set_ylim(0, max(total_all_arr) * 1.15)
ax.set_xlabel("MGA slack (fraction, ε)", fontsize=12)
ax.set_ylabel("HBI shipping trade (Mt)", fontsize=12)
ax.set_title("HBI Shipping Trade Through Maritime Chokepoints vs MGA Slack (mga-chokepoints)", fontsize=13)
ax.grid(True, linestyle="--", linewidth=0.5, alpha=0.5)
ax.legend(loc="best", fontsize=8, framealpha=0.95, ncol=2, title="Chokepoint")

plt.tight_layout()
plt.show()

# Print summary
if total_all_arr[0] > 0:
    print(f"\nTotal (all chokepoints, summed): {total_all_arr[0]:.1f} Mt (optimal) → {total_all_arr[-1]:.1f} Mt (ε={epsilon_arr[-1]})")
    print(f"  Change: {((total_all_arr[-1] / total_all_arr[0]) - 1) * 100:.1f}%")
if total_any_arr[0] > 0:
    print(f"Total (any chokepoint, deduplicated): {total_any_arr[0]:.1f} Mt (optimal) → {total_any_arr[-1]:.1f} Mt (ε={epsilon_arr[-1]})")
    print(f"  Change: {((total_any_arr[-1] / total_any_arr[0]) - 1) * 100:.1f}%")


---
# MGA Analysis: Inter-Block HBI Trade

Analyse how much HBI shipping trade flows **between** the two geopolitical blocks defined in the **mga-blocks** scenario. Trade *within* the same block is unconstrained; only inter-block trade is minimised via MGA.

### Load blocks configuration and networks

In [ ]:
# ── Load MGA-blocks scenario configuration ──
# bl_scenario is derived from snakemake.input.network_mga_blocks in the config cell above
bl_mga_config = snakemake.config.get("scenario", {}).get(bl_scenario, {})
bl_mga_modifiers = bl_mga_config.get("modifiers", {})
bl_mga_slack_values = bl_mga_modifiers.get("mga", {}).get("slack", [])
bl_carrier = bl_mga_modifiers.get("mga", {}).get("carrier", "hbi")

# Block definitions from config.
# Each entry in the YAML list may be a comma-separated string, so split defensively.
def _parse_block_regions(region_list):
    """Flatten potentially comma-separated strings within a YAML list."""
    result = []
    for entry in (region_list or []):
        for r in str(entry).split(","):
            r = r.strip()
            if r:
                result.append(r)
    return result

bl_threshold = bl_mga_modifiers.get("mga", {}).get("threshold_value", {})
block_1_regions = _parse_block_regions(bl_threshold.get("block_1", []))
block_2_regions = _parse_block_regions(bl_threshold.get("block_2", []))
block_3_regions = _parse_block_regions(bl_threshold.get("block_3", []))

print(f"Blocks MGA scenario: {bl_scenario}")
print(f"Slack values: {bl_mga_slack_values}")
print(f"Block 1 ({len(block_1_regions)} regions): {block_1_regions}")
print(f"Block 2 ({len(block_2_regions)} regions): {block_2_regions}")
print(f"Block 3 ({len(block_3_regions)} regions): {block_3_regions}")

# Build region → block mapping
region_to_block = {}
for r in block_1_regions:
    region_to_block[r] = "block_1"
for r in block_2_regions:
    region_to_block[r] = "block_2"
for r in block_3_regions:
    region_to_block[r] = "block_3"

# ── Load PyPSA networks from mga-blocks results ──
bl_results_dir = os.path.abspath(os.path.join(
    os.path.dirname(snakemake.input.network_mga_blocks)
))

print(f"\nResults directory: {bl_results_dir}")

bl_networks_dict = {}
bl_slack_values = [0] + bl_mga_slack_values

for slack in bl_slack_values:
    if slack == 0:
        filepath = os.path.join(bl_results_dir, "network_nan.nc")
        slack_key = None
    else:
        filepath = os.path.join(bl_results_dir, f"network_{slack}.nc")
        slack_key = slack

    if os.path.exists(filepath):
        try:
            bl_networks_dict[slack_key] = pypsa.Network(filepath)
            label = "optimal (slack=0)" if slack == 0 else f"slack={slack}"
            print(f"  ✓ Loaded {label}")
        except Exception as e:
            print(f"  ✗ Error loading {filepath}: {e}")
    else:
        print(f"  ✗ File not found: {filepath}")

print(f"\n✓ Loaded {len(bl_networks_dict)} blocks MGA networks")

# ── Sort keys and build epsilon array ──
bl_sorted_keys = sorted(bl_networks_dict.keys(), key=lambda x: (x is not None, x))
bl_epsilon_values = [0 if k is None else k for k in bl_sorted_keys]


### Extract inter-block HBI trade per MGA slack

In [ ]:
# For each blocks MGA network, classify HBI shipping links as inter-block
# or intra-block and sum trade volumes.

carrier = bl_carrier
BLOCKS = ["block_1", "block_2", "block_3"]

# All 6 directional inter-block pairs
bl_interblock_total = []
bl_intrablock_total = []
# keyed as "block_X → block_Y"
pair_series = {f"{src} → {dst}": [] for src in BLOCKS for dst in BLOCKS if src != dst}

for net_key in bl_sorted_keys:
    n_bl = bl_networks_dict[net_key]

    inter_total = 0.0
    intra_total = 0.0
    pair_totals = {k: 0.0 for k in pair_series}

    shipping_links = n_bl.links[n_bl.links.carrier == f"shipping_{carrier}"]

    for link_name, link_row in shipping_links.iterrows():
        bus0 = link_row["bus0"]
        bus1 = link_row["bus1"]
        r_from = bus0.rsplit(f"_{carrier}", 1)[0]
        r_to = bus1.rsplit(f"_{carrier}", 1)[0]

        trade_vol = link_row.get("p_nom_opt", 0)
        if pd.isna(trade_vol) or trade_vol <= 0:
            continue

        block_from = region_to_block.get(r_from)
        block_to = region_to_block.get(r_to)

        if block_from and block_to and block_from != block_to:
            inter_total += trade_vol
            pair_key = f"{block_from} → {block_to}"
            if pair_key in pair_totals:
                pair_totals[pair_key] += trade_vol
        else:
            intra_total += trade_vol

    bl_interblock_total.append(inter_total)
    bl_intrablock_total.append(intra_total)
    for k in pair_series:
        pair_series[k].append(pair_totals[k])

# Build summary DataFrame
summary_dict = {"Inter-block total": bl_interblock_total}
for k, vals in pair_series.items():
    # humanise key: "block_1 → block_2" → "Block 1 → Block 2"
    label = k.replace("block_", "Group ").replace("_", " ")
    summary_dict[label] = vals
summary_dict["Intra-block total"] = bl_intrablock_total

bl_summary = pd.DataFrame(summary_dict, index=bl_epsilon_values) / 1e6  # Mt
bl_summary.index.name = "MGA Slack (ε)"
print("HBI shipping trade by block classification (Mt):")
bl_summary.round(1)


### Plot: Inter-block HBI trade vs MGA slack

In [ ]:
bl_epsilon_arr = np.array(bl_epsilon_values, dtype=float)

inter_arr = np.array(bl_summary["Inter-block total"].values, dtype=float)
intra_arr = np.array(bl_summary["Intra-block total"].values, dtype=float)

# Bloc colors from config
_bl_c1 = snakemake.config["colors"].get("bloc_1", "#17679E")
_bl_c2 = snakemake.config["colors"].get("bloc_2", "#8352D8")
_bl_c3 = snakemake.config["colors"].get("bloc_3", "#BF9A39")

# All 6 directional pairs with distinct colours / styles
pair_plot_cfg = {
    "Block 1 → Block 2": dict(color=_bl_c1, linestyle="--", marker="s"),
    "Block 1 → Block 3": dict(color=_bl_c1, linestyle=":",  marker="^"),
    "Block 2 → Block 1": dict(color=_bl_c2, linestyle="--", marker="s"),
    "Block 2 → Block 3": dict(color=_bl_c2, linestyle=":",  marker="^"),
    "Block 3 → Block 1": dict(color=_bl_c3, linestyle="--", marker="s"),
    "Block 3 → Block 2": dict(color=_bl_c3, linestyle=":",  marker="^"),
}

fig, ax = plt.subplots(1, 1, figsize=(12, 7))

# ── Total inter-block trade (thick black) ──
ax.plot(bl_epsilon_arr, inter_arr, color="black", linewidth=3, marker="o",
        markersize=8, label="Inter-block total", zorder=20)

# ── All 6 directional pairs ──
for label, cfg in pair_plot_cfg.items():
    if label in bl_summary.columns:
        vals = np.array(bl_summary[label].values, dtype=float)
        ax.plot(bl_epsilon_arr, vals,
                color=cfg["color"], linewidth=1.5, linestyle=cfg["linestyle"],
                marker=cfg["marker"], markersize=5, alpha=0.8,
                label=label, zorder=10)

# ── Intra-block trade (dotted grey) ──
ax.plot(bl_epsilon_arr, intra_arr, color="#999999", linewidth=1.5,
        linestyle=":", marker="d", markersize=4, alpha=0.7,
        label="Intra-block total", zorder=5)

# ── Axes formatting ──
ax.set_xlim(min(bl_epsilon_arr) - 0.002, max(bl_epsilon_arr) + 0.01)
y_max = max(inter_arr.max(), intra_arr.max()) * 1.15
ax.set_ylim(0, y_max)
ax.set_xlabel("MGA slack (fraction, ε)", fontsize=12)
ax.set_ylabel("HBI shipping trade (Mt)", fontsize=12)
ax.set_title("Inter-Block vs Intra-Block HBI Trade vs MGA Slack (mga-blocs)", fontsize=13)
ax.grid(True, linestyle="--", linewidth=0.5, alpha=0.5)
ax.legend(loc="best", fontsize=9, framealpha=0.95, title="Trade classification")

plt.tight_layout()
plt.show()

# Print summary
if inter_arr[0] > 0:
    print(f"\nInter-block trade: {inter_arr[0]:.1f} Mt (optimal) → {inter_arr[-1]:.1f} Mt (ε={bl_epsilon_arr[-1]})")
    print(f"  Change: {((inter_arr[-1] / inter_arr[0]) - 1) * 100:.1f}%")
print(f"Intra-block trade: {intra_arr[0]:.1f} Mt (optimal) → {intra_arr[-1]:.1f} Mt (ε={bl_epsilon_arr[-1]})")

# Block definitions legend
print(f"\nBlock 1: {', '.join(block_1_regions)}")
print(f"Block 2: {', '.join(block_2_regions)}")
print(f"Block 3: {', '.join(block_3_regions)}")


---
# Combined MGA Overview: Three Subplots

Bring all three MGA investigations into a single figure, following the style of the conceptual "Plot 1B" template. Uses the already-prepared variables `df_cp_display`, `df_hbi`, and `bl_summary`.

In [ ]:

# ═══════════════════════════════════════════════════════════════════════════════
# 1) Chokepoint data  (from df_cp / df_cp_display)
# ═══════════════════════════════════════════════════════════════════════════════
# Total chokepoint-crossing exposure (Mt) – TOTAL (all chokepoints) row
# A route through N chokepoints is counted N times.
cp_total_row = df_cp_display.loc["TOTAL (all chokepoints)"]
cp_eps = np.array([float(c) for c in cp_total_row.index]) * 100  # → %
cp_total_vals = np.array(cp_total_row.values, dtype=float)

# Per-chokepoint breakdown (only those with any trade), sorted by final value descending
cp_region_lines = []
for cpname in config_chokepoints:
    if cpname in df_cp.index and df_cp.loc[cpname].sum() > 0:
        vals = np.array(df_cp.loc[cpname].values, dtype=float)
        cp_region_lines.append(dict(
            name=cp_display_names.get(cpname, cpname.replace("_", " ").title()),
            values=vals,
            color=cp_colors.get(cpname, "#333333"),
            linestyle=cp_linestyles.get(cpname, "-"),
            alpha=0.8 if cpname in ("babalmandab", "suez") else 0.6,
        ))
cp_region_lines.sort(key=lambda d: d["values"][-1], reverse=True)

# ═══════════════════════════════════════════════════════════════════════════════
# 2) Stable exporters data  (from df_hbi)
# ═══════════════════════════════════════════════════════════════════════════════
stab_eps_cols = df_hbi.columns[:-2]  # all except metadata cols
stab_eps = np.array([float(c) for c in stab_eps_cols]) * 100  # → %

stab_stable_total = np.array(
    df_hbi[df_hbi["Stability Class"] == "Stable"][stab_eps_cols].sum().values, dtype=float
)
stab_unstable_total = np.array(
    df_hbi[df_hbi["Stability Class"] == "Unstable"][stab_eps_cols].sum().values, dtype=float
)

# Per-region breakdown, sorted by final value descending within each group
stab_region_lines_stable = []
for region in stable_regions:
    if region in df_hbi.index:
        vals = np.array(df_hbi.loc[region, stab_eps_cols].values, dtype=float)
        if vals.sum() > 0:
            stab_region_lines_stable.append(dict(
                name=region_nice_names.get(region, region.replace("_", " ")),
                values=vals,
                color=region_colors.get(region, "#333333"),
                linestyle="-",
                alpha=0.6,
            ))
stab_region_lines_stable.sort(key=lambda d: d["values"][-1], reverse=True)

stab_region_lines_unstable = []
for region in unstable_regions:
    if region in df_hbi.index:
        vals = np.array(df_hbi.loc[region, stab_eps_cols].values, dtype=float)
        if vals.sum() > 0:
            stab_region_lines_unstable.append(dict(
                name=region_nice_names.get(region, region.replace("_", " ")),
                values=vals,
                color=region_colors.get(region, "#333333"),
                linestyle="--",
                alpha=0.7,
            ))
stab_region_lines_unstable.sort(key=lambda d: d["values"][-1], reverse=True)

# ═══════════════════════════════════════════════════════════════════════════════
# 3) Blocs data  (from bl_summary)
# ═══════════════════════════════════════════════════════════════════════════════
bl_eps = np.array([float(c) for c in bl_summary.index]) * 100  # → %
bl_inter_vals = np.array(bl_summary["Inter-block total"].values, dtype=float)
bl_intra_vals = np.array(bl_summary["Intra-block total"].values, dtype=float)

# All 6 directional pairs with consistent colours per source bloc (from config)
_bl_c1 = snakemake.config["colors"].get("bloc_1", "#17679E")
_bl_c2 = snakemake.config["colors"].get("bloc_2", "#8352D8")
_bl_c3 = snakemake.config["colors"].get("bloc_3", "#BF9A39")
_bl_pair_cfg = {
    "Block 1 → Block 2": dict(color=_bl_c1, linestyle="--"),
    "Block 1 → Block 3": dict(color=_bl_c1, linestyle=":"),
    "Block 2 → Block 1": dict(color=_bl_c2, linestyle="--"),
    "Block 2 → Block 3": dict(color=_bl_c2, linestyle=":"),
    "Block 3 → Block 1": dict(color=_bl_c3, linestyle="--"),
    "Block 3 → Block 2": dict(color=_bl_c3, linestyle=":"),
}
bl_region_lines_unsorted = []
for label, cfg in _bl_pair_cfg.items():
    if label in bl_summary.columns:
        bl_region_lines_unsorted.append(dict(
            name=label,
            values=np.array(bl_summary[label].values, dtype=float),
            color=cfg["color"],
            linestyle=cfg["linestyle"],
            alpha=0.7,
        ))
bl_region_lines = sorted(bl_region_lines_unsorted, key=lambda d: d["values"][-1], reverse=False)

# ═══════════════════════════════════════════════════════════════════════════════
# Plot definitions
# ═══════════════════════════════════════════════════════════════════════════════
Y_MAX = 800  # common y-axis limit for all subplots

plots = [
    dict(
        title="Unstable trade routes (chokepoints)",
        epsilon=cp_eps,
        main_curve=cp_total_vals,
        y_mid=cp_total_vals[0],
        color="#185FA5",
        alpha=0.25,
        arrow_up=None,
        arrow_dn=["Decreased", "high-risk routes"],
        arrow_x_frac=0.70,
        y_label_add="trade through maritime chokepoints",
        regions=cp_region_lines,
        extra_curves=[],
    ),
    dict(
        title="Production in unstable countries",
        epsilon=stab_eps,
        main_curve=stab_unstable_total,
        y_mid=stab_unstable_total[0],
        color="#3B6D11",
        alpha=0.25,
        arrow_up=None,
        arrow_dn=["Decreased unstable", "production"],
        arrow_x_frac=0.45,
        y_label_add="production in stable regions",
        regions=stab_region_lines_stable + stab_region_lines_unstable,
        extra_curves=[],
        # Legend grouping: n_stable stable lines, then n_unstable unstable lines
        n_stable=len(stab_region_lines_stable),
        n_unstable=len(stab_region_lines_unstable),
    ),
    dict(
        title="Fragmentation / Bloc trade",
        epsilon=bl_eps,
        main_curve=bl_inter_vals,
        y_mid=bl_inter_vals[0],
        color="#534AB7",
        alpha=0.25,
        arrow_up=None,
        arrow_dn=["Decreased trade", "between blocs"],
        arrow_x_frac=0.70,
        y_label_add="trade between blocs",
        regions=bl_region_lines,
        extra_curves=[
            dict(name="Intra-bloc total", values=bl_intra_vals,
                 color="#999999", linestyle=":", linewidth=1.5, alpha=0.7),
        ],
    ),
]

# ═══════════════════════════════════════════════════════════════════════════════
# Layout
# ═══════════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.subplots_adjust(wspace=0.35)

GAP_FRAC = 0.04

for ax, p in zip(axes, plots):
    eps = p["epsilon"]
    curve = p["main_curve"]
    y_mid = p["y_mid"]
    gap = Y_MAX * GAP_FRAC

    # Main curve
    ax.plot(eps, curve, color=p["color"], linewidth=2.5, marker="o",
            markersize=4, label="Global", zorder=10)

    # Extra curves (e.g. intra-bloc)
    for ec in p.get("extra_curves", []):
        ax.plot(eps, ec["values"], color=ec["color"],
                linewidth=ec.get("linewidth", 1.2),
                linestyle=ec["linestyle"], alpha=ec["alpha"],
                marker="d", markersize=3, label=ec["name"], zorder=5)

    # Regional decomposition lines
    if p.get("regions"):
        for reg in p["regions"]:
            ax.plot(eps, reg["values"], color=reg["color"], linewidth=1.2,
                    linestyle=reg["linestyle"], alpha=reg["alpha"],
                    label=reg["name"], zorder=5)

    # ── Axes formatting ──
    ax.set_xlim(eps[0], eps[-1])
    ax.set_ylim(0, Y_MAX)
    ax.set_xlabel("ε in %", fontsize=11)
    ax.set_ylabel(f"HBI {p['y_label_add']} in Mt", fontsize=10)
    ax.set_title(p["title"], fontsize=13, fontweight="medium", pad=10)
    ax.grid(True, linestyle="--", linewidth=0.5, alpha=0.5)

    # ── Legend: Global always on top; within groups, lines sorted by value (high→low) ──
    handles, labels = ax.get_legend_handles_labels()
    if "n_stable" in p:
        # Production subplot: Global | stable group (pre-sorted) | unstable group (pre-sorted)
        n_s = p["n_stable"]
        global_h, global_l     = handles[:1],      labels[:1]
        stable_h, stable_l     = handles[1:1+n_s], labels[1:1+n_s]
        unstable_h, unstable_l = handles[1+n_s:],  labels[1+n_s:]
        ax.legend(global_h + stable_h + unstable_h,
                  global_l + stable_l + unstable_l,
                  loc="best", fontsize=7, framealpha=0.9)
    else:
        # Global on top, remaining entries reversed (highest value at top)
        ax.legend([handles[0]] + handles[1:][::-1],
                  [labels[0]] + labels[1:][::-1],
                  loc="best", fontsize=7, framealpha=0.9)

    # ── Arrows ──
    arrowprops = dict(arrowstyle="->", color="#444441", lw=1.4, mutation_scale=12)

    arrow_x_frac = p.get("arrow_x_frac", 0.70)
    arrow_x = eps[0] + arrow_x_frac * (eps[-1] - eps[0])
    label_x = eps[0] + (arrow_x_frac + 0.05) * (eps[-1] - eps[0])

    curve_at_arrow = np.interp(arrow_x, eps, curve)

    if p["arrow_dn"] is not None:
        y_start = y_mid - gap
        y_end = curve_at_arrow
        if y_end < y_start:
            ax.annotate("", xy=(arrow_x, y_end), xytext=(arrow_x, y_start),
                        xycoords="data", textcoords="data",
                        arrowprops=arrowprops, clip_on=False)
            y_label = (y_start + y_end) / 2
            ax.text(label_x, y_label, "\n".join(p["arrow_dn"]),
                    fontsize=8, va="center", ha="left", color="#444441", clip_on=False)

    if p["arrow_up"] is not None:
        y_start = y_mid + gap
        y_end = curve_at_arrow
        if y_end > y_start:
            ax.annotate("", xy=(arrow_x, y_end), xytext=(arrow_x, y_start),
                        xycoords="data", textcoords="data",
                        arrowprops=arrowprops, clip_on=False)
            y_label = (y_start + y_end) / 2
            ax.text(label_x, y_label, "\n".join(p["arrow_up"]),
                    fontsize=8, va="center", ha="left", color="#444441", clip_on=False)

plt.tight_layout()
plt.show()


---
# Supply Constraint Pareto: Per-Region HBI Production vs Cost Premium

Load the `constrain-supply` scenario networks, compute the cost premium (ε) relative to the unconstrained optimum for each per-region supply cap, and extract regional HBI production — analogous to the MGA stability analysis above but driven by supply-cap tightness rather than MGA slack.


In [ ]:

# ── Load Pareto Supply Constraint networks ────────────────────────────────────
pareto_scenario_name = extract_scenario(snakemake.input.network_pareto_supply)
pareto_cfg = snakemake.config.get("scenario", {}).get(pareto_scenario_name, {})
pareto_limits_cfg = (
    pareto_cfg.get("modifiers", {})
    .get("pareto", {})
    .get("threshold_value", [])
)

pareto_dir = os.path.dirname(snakemake.input.network_pareto_supply)
print(f"Pareto supply directory: {pareto_dir}")
print(f"Configured per-region caps (Mt): {pareto_limits_cfg}")

# Optimal (unconstrained) — same as network.nc but stored as network_nan.nc in the collection
opt_path = os.path.join(pareto_dir, "network_nan.nc")
if not os.path.exists(opt_path):
    opt_path = snakemake.input.network_pareto_supply   # fallback: network.nc

pareto_nets = {}
pareto_nets[None] = pypsa.Network(opt_path)
print("  ✓ Loaded optimal (unconstrained)")

for lim in pareto_limits_cfg:
    path = os.path.join(pareto_dir, f"network_{float(lim)}.nc")
    if os.path.exists(path):
        pareto_nets[float(lim)] = pypsa.Network(path)
        print(f"  ✓ Loaded limit = {lim} Mt")
    else:
        print(f"  ✗ Not found: {path}")

# ── Compute total system cost for each network ────────────────────────────────
def total_system_cost_bn(n):
    """Return total system cost in billion EUR."""
    tsc = (
        pd.concat([n.statistics.capex(), n.statistics.opex()], axis=1)
        .sum(axis=1)
        .div(1e9)
    )
    return float(tsc.sum())

optimal_cost_pareto = total_system_cost_bn(pareto_nets[None])
print(f"\nOptimal system cost: {optimal_cost_pareto:.3f} B€")

# Compute ε = (cost_constrained − cost_optimal) / cost_optimal for every network,
# then sort by ε ascending so the x-axis runs from 0 (unconstrained) to max ε.
pareto_entries = [(None, 0.0)]
for key in [k for k in pareto_nets if k is not None]:
    cost = total_system_cost_bn(pareto_nets[key])
    eps = (cost - optimal_cost_pareto) / optimal_cost_pareto
    pareto_entries.append((key, eps))

pareto_entries.sort(key=lambda x: x[1])   # sort by ε ascending

# ── Deduplicate entries with identical ε ──────────────────────────────────────
# Non-binding constraints (e.g. a 400 Mt cap when optimal production is already
# below 400 Mt everywhere) produce the same cost as the unconstrained optimal,
# yielding ε = 0 for multiple entries.  Duplicate ε values would create duplicate
# DataFrame column labels, causing pandas .loc to return more values than expected.
# Keep the first occurrence (= unconstrained optimal for ε=0 ties).
seen_eps: set = set()
deduped: list = []
for key, eps in pareto_entries:
    eps_key = round(eps, 8)   # round to absorb floating-point noise
    if eps_key not in seen_eps:
        seen_eps.add(eps_key)
        deduped.append((key, eps))
    else:
        label = "unconstrained" if key is None else f"{key:.0f} Mt cap"
        print(f"  (skipped duplicate ε={eps*100:.3f}%: {label})")
pareto_entries = deduped

ordered_pareto_keys = [e[0] for e in pareto_entries]
pareto_eps_arr_frac  = np.array([e[1] for e in pareto_entries], dtype=float)

for key, eps in pareto_entries:
    label = "unconstrained" if key is None else f"{key:.0f} Mt cap"
    cost  = total_system_cost_bn(pareto_nets[key])
    print(f"  {label:25s}: cost = {cost:.3f} B€  →  ε = {eps * 100:.2f}%")

# ── Extract HBI production per region ─────────────────────────────────────────
# Reuse all_regions / stable_regions / unstable_regions from the stability cell above
pareto_hbi_data = {region: [] for region in all_regions}

for key in ordered_pareto_keys:
    n_p = pareto_nets[key]
    regional_totals = {region: 0.0 for region in all_regions}

    for link_idx, link_row in n_p.links.iterrows():
        if link_row["carrier"] == "hbi":
            bus1 = link_row["bus1"]
            if hasattr(n_p, "links_t") and "p0" in n_p.links_t:
                p_sum = n_p.links_t["p0"][link_idx].sum()
            else:
                p_sum = link_row.get("p_nom_opt", 0.0)

            if not pd.isna(p_sum) and p_sum > 0:
                for region in all_regions:
                    if bus1.startswith(region):
                        regional_totals[region] += p_sum
                        break

    for region in all_regions:
        pareto_hbi_data[region].append(regional_totals[region])

# Build DataFrame: regions × ε values (using integer positions as column names
# to avoid duplicate-label issues when several caps produce identical ε)
n_scenarios = len(pareto_eps_arr_frac)
df_pareto_hbi = pd.DataFrame(
    pareto_hbi_data,
    index=range(n_scenarios)
).T / 1e6
df_pareto_hbi = df_pareto_hbi / snakemake.config["iron_ore"]["ore_to_steel_ratio"]
df_pareto_hbi.index.name   = "Region"
df_pareto_hbi.columns.name = "Scenario index"

# Attach political-stability metadata (for consistent colour/style coding)
df_pareto_hbi["Political Stability"] = stability_df.iloc[:, 0]
df_pareto_hbi["Stability Class"] = df_pareto_hbi.index.map(
    lambda r: "Stable" if r in stable_regions else "Unstable"
)

pareto_eps_cols = df_pareto_hbi.columns[:-2]   # integer scenario indices (0, 1, 2, ...)

print(f"\n{len(pareto_entries)} unique scenarios after deduplication")
print("\nHBI Production summary (Mt):")
df_pareto_hbi[pareto_eps_cols].round(1)


---
# Combined Overview Plot (v2): Chokepoints | Supply Constraint | Blocs

Same three-panel layout as above, but the centre panel now shows the **supply-constraint Pareto front** — how HBI production redistributes across regions as per-region production is progressively capped (x-axis = cost premium ε derived from each constrained optimisation).


In [ ]:
steel_demand = pd.read_csv(snakemake.input.steel_demand, index_col="region")
steel_demand = steel_demand["SteelDemand_DRI_Mt"].sum()

In [ ]:

# ═══════════════════════════════════════════════════════════════════════════════
# 1) Chokepoint data  — reused from df_cp / df_cp_display (computed above)
# ═══════════════════════════════════════════════════════════════════════════════
cp_total_row_v2 = df_cp_display.loc["TOTAL (all chokepoints)"]
cp_eps_v2       = np.array([float(c) for c in cp_total_row_v2.index]) * 100   # → %
cp_total_vals_v2 = np.array(cp_total_row_v2.values, dtype=float)

cp_region_lines_v2 = []
for cpname in config_chokepoints:
    if cpname in df_cp.index and df_cp.loc[cpname].sum() > 0:
        vals = np.array(df_cp.loc[cpname].values, dtype=float)
        cp_region_lines_v2.append(dict(
            name=cp_display_names.get(cpname, cpname.replace("_", " ").title()),
            values=vals,
            color=cp_colors.get(cpname, "#333333"),
            linestyle=cp_linestyles.get(cpname, "-"),
            alpha=0.8 if cpname in ("babalmandab", "suez") else 0.6,
        ))
cp_region_lines_v2.sort(key=lambda d: d["values"][-1], reverse=True)

# ═══════════════════════════════════════════════════════════════════════════════
# 2) Supply constraint data  — from df_pareto_hbi / pareto_eps_arr_frac
#
#    pareto_eps_cols : integer position labels (0, 1, 2, …) — safe for .loc
#    sc_eps_pct      : actual ε values in % (from pareto_eps_arr_frac)
#    sc_cap_vals     : "max. per supplier" curve
#                      • unconstrained optimum (key=None, ε=0): max configured
#                        cap value from config (pareto_limits_cfg)
#                      • constrained scenarios: raw threshold_value from config
#                        (Mt, as specified — no unit conversion applied)
# ═══════════════════════════════════════════════════════════════════════════════
sc_eps_pct = pareto_eps_arr_frac * 100   # → %

# Build cap line directly from config threshold values (ordered by ε ascending,
# matching the column order of df_pareto_hbi).
# For the unconstrained optimal (key=None, ε=0), use the maximum configured cap
# value so the curve starts at the "no constraint" level.
sc_cap_vals = np.array([
    float(max(float(x) for x in pareto_limits_cfg)) if key is None
    else float(key)
    for key in ordered_pareto_keys
], dtype=float)
print(f"max. per supplier line (cap values): {sc_cap_vals}")

# Individual region lines — all solid (stable/unstable not distinguished here)
sc_region_lines = []
for region in all_regions:
    vals = np.array(df_pareto_hbi.loc[region, pareto_eps_cols].values, dtype=float)
    if vals.sum() > 0:
        sc_region_lines.append(dict(
            name=region_nice_names.get(region, region.replace("_", " ")),
            values=vals,
            color=region_colors.get(region, "#333333"),
            linestyle="-",
            alpha=0.65,
        ))
# Sort by production at ε=0 descending so the legend lists largest producers first
sc_region_lines.sort(key=lambda d: d["values"][0], reverse=True)

# Total production across all regions (approximately constant = demand-driven)
sc_total_vals = np.array(df_pareto_hbi[pareto_eps_cols].sum().values, dtype=float)

assert len(sc_eps_pct) == len(sc_cap_vals), (
    f"eps ({len(sc_eps_pct)}) and cap curve ({len(sc_cap_vals)}) length mismatch"
)

# ═══════════════════════════════════════════════════════════════════════════════
# 3) Blocs data  — reused from bl_summary (computed above)
# ═══════════════════════════════════════════════════════════════════════════════
bl_eps_v2       = np.array([float(c) for c in bl_summary.index]) * 100   # → %
bl_inter_vals_v2 = np.array(bl_summary["Inter-block total"].values, dtype=float)
bl_intra_vals_v2 = np.array(bl_summary["Intra-block total"].values, dtype=float)

# Bloc colors from config
_bl_c1_v2 = snakemake.config["colors"].get("bloc_1", "#17679E")
_bl_c2_v2 = snakemake.config["colors"].get("bloc_2", "#8352D8")
_bl_c3_v2 = snakemake.config["colors"].get("bloc_3", "#BF9A39")
_bl_pair_cfg_v2 = {
    "Group 1 → Group 2": dict(color=_bl_c1_v2, linestyle="--"),
    "Group 1 → Group 3": dict(color=_bl_c1_v2, linestyle=":"),
    "Group 2 → Group 1": dict(color=_bl_c2_v2, linestyle="--"),
    "Group 2 → Group 3": dict(color=_bl_c2_v2, linestyle=":"),
    "Group 3 → Group 1": dict(color=_bl_c3_v2, linestyle="--"),
    "Group 3 → Group 2": dict(color=_bl_c3_v2, linestyle=":"),
}
bl_region_lines_v2 = sorted(
    [
        dict(name=label, values=np.array(bl_summary[label].values, dtype=float),
             color=cfg["color"], linestyle=cfg["linestyle"], alpha=0.7)
        for label, cfg in _bl_pair_cfg_v2.items()
        if label in bl_summary.columns
    ],
    key=lambda d: d["values"][-1],
    reverse=False,
)

# ═══════════════════════════════════════════════════════════════════════════════
# Convert absolute values (Mt) to relative values (% of global steel demand)
# ═══════════════════════════════════════════════════════════════════════════════
def _scale_lines(lines, factor):
    """Return a copy of a list of region-line dicts with values scaled by factor."""
    return [{**r, "values": r["values"] * factor} for r in lines]

_pct_factor = 100.0 / steel_demand   # Mt → % of market share

cp_total_vals_v2_pct   = cp_total_vals_v2 * _pct_factor
cp_region_lines_v2_pct = _scale_lines(cp_region_lines_v2, _pct_factor)

bl_inter_vals_v2_pct   = bl_inter_vals_v2 * _pct_factor
bl_intra_vals_v2_pct   = bl_intra_vals_v2 * _pct_factor
bl_region_lines_v2_pct = _scale_lines(bl_region_lines_v2, _pct_factor)

sc_cap_vals_pct        = sc_cap_vals * _pct_factor
sc_region_lines_pct    = _scale_lines(sc_region_lines, _pct_factor)

# ═══════════════════════════════════════════════════════════════════════════════
# Combined plot — v2
# ═══════════════════════════════════════════════════════════════════════════════
Y_MAX_V2 = 100
GAP_FRAC_V2 = 0.04

plots_v2 = [
    dict(
        title="Stable trade routes (chokepoints)",
        epsilon=cp_eps_v2,
        main_curve=cp_total_vals_v2_pct,
        main_label="Global",
        y_mid=cp_total_vals_v2_pct[0],
        color="#000000",
        arrow_dn=["Minimize", "high-risk routes"],
        arrow_x_frac=0.40,
        y_label_add="trade through maritime chokepoints \nin % of market share",
        regions=cp_region_lines_v2_pct,
        extra_curves=[],
    ),
    dict(
        title="Friendshoring",
        epsilon=bl_eps_v2,
        main_curve=bl_inter_vals_v2_pct,
        main_label="Global",
        y_mid=bl_inter_vals_v2_pct[0],
        color="#000000",
        arrow_dn=["Minimize trade with", "non-friendshored regions"],
        arrow_x_frac=0.40,
        y_label_add="trade between groups \nin % of market share",
        regions=bl_region_lines_v2_pct,
        extra_curves=[
            dict(name="Friendshoring total", values=bl_intra_vals_v2_pct,
                 color="#999999", linestyle=":", linewidth=1.5, alpha=0.7),
        ],
    ),
    dict(
        title="Supply diversification",
        epsilon=sc_eps_pct,
        main_curve=sc_cap_vals_pct,
        main_label="max. per supplier",
        y_mid=sc_cap_vals_pct[0],
        color="#000000",
        arrow_dn=["Constrain production", "per supplier"],
        arrow_x_frac=0.30,
        y_label_add="production by region \nin % of market share",
        regions=sc_region_lines_pct,
        extra_curves=[],
    ),
]

fig_v2, axes_v2 = plt.subplots(1, 3, figsize=(15, 5))
fig_v2.subplots_adjust(wspace=0.35)

arrowprops_v2 = dict(arrowstyle="->", color="#444441", lw=1.4, mutation_scale=12)

for ax, p in zip(axes_v2, plots_v2):
    eps   = p["epsilon"]
    curve = p["main_curve"]
    y_mid = p["y_mid"]
    gap   = Y_MAX_V2 * GAP_FRAC_V2

    # ── Main curve ───────────────────────────────────────────────────────────
    ax.plot(eps, curve, color=p["color"], linewidth=1.5, marker="o",markersize=3, label=p["main_label"], zorder=0, linestyle="-")

    # ── Extra reference curves ───────────────────────────────────────────────
    for ec in p.get("extra_curves", []):
        ax.plot(eps, ec["values"],
                color=ec["color"],
                linewidth=ec.get("linewidth", 1.2),
                linestyle=ec["linestyle"],
                alpha=ec["alpha"],
                marker="d", markersize=3,
                label=ec["name"], zorder=5)

    # ── Per-region decomposition lines ───────────────────────────────────────
    for reg in p.get("regions", []):
        ax.plot(eps, reg["values"],
                color=reg["color"], linewidth=1.2,
                linestyle=reg["linestyle"], alpha=reg["alpha"],
                label=reg["name"], zorder=5)

    # ── Axes ─────────────────────────────────────────────────────────────────
    ax.set_xlim(eps[0], eps[-1])
    ax.set_ylim(0, Y_MAX_V2)
    ax.set_yticks(np.arange(0, Y_MAX_V2 + 1, 10))
    ax.set_xlabel("ε in %", fontsize=11)
    ax.set_ylabel(f"HBI {p['y_label_add']}", fontsize=10, labelpad=-2)
    ax.set_title(p["title"], fontsize=13, fontweight="medium", pad=10)
    ax.grid(True, linestyle="--", linewidth=0.5, alpha=0.5)

    # ── Legend: main curve first, then remaining reversed (highest value on top) ──
    handles, labels = ax.get_legend_handles_labels()
    ax.legend(
        [handles[0]] + handles[1:][::-1],
        [labels[0]]  + labels[1:][::-1],
        loc="best", fontsize=7, framealpha=0.9,
    )

    # ── Downward arrow ───────────────────────────────────────────────────────
    if p.get("arrow_dn") is not None:
        arrow_x_frac = p.get("arrow_x_frac", 0.70)
        arrow_x = eps[0] + arrow_x_frac * (eps[-1] - eps[0])
        label_x = eps[0] + (arrow_x_frac + 0.05) * (eps[-1] - eps[0])
        curve_at_arrow = np.interp(arrow_x, eps, curve)
        y_start = y_mid - gap
        y_end   = curve_at_arrow
        if y_end < y_start:
            ax.annotate("", xy=(arrow_x, y_end), xytext=(arrow_x, y_start),
                        xycoords="data", textcoords="data",
                        arrowprops=arrowprops_v2, clip_on=False)
            ax.text(label_x, (y_start + y_end) / 2, "\n".join(p["arrow_dn"]),
                    fontsize=8, va="center", ha="left", color="#444441", clip_on=False)

plt.tight_layout()

plt.savefig(snakemake.output.mga_plot, dpi=300, bbox_inches="tight")
plt.savefig(snakemake.output.mga_plot_png, dpi=300, bbox_inches="tight")
plt.show()

# ── Summary stats ─────────────────────────────────────────────────────────────
print(f"\nmax. per supplier cap: {sc_cap_vals[0]:.1f} Mt (ε=0) → {sc_cap_vals[-1]:.1f} Mt (ε={sc_eps_pct[-1]:.2f}%)")
print(f"Total production:      {sc_total_vals[0]:.1f} Mt (ε=0) → {sc_total_vals[-1]:.1f} Mt (ε={sc_eps_pct[-1]:.2f}%)")


# Misc

In [ ]:
n.statistics.energy_balance().div(1e6)

In [ ]:
n = cp_networks_dict[None] # None for optimal solution, or use specific slack value if desired

In [ ]:
# Total shipping volume
n.links[n.links.carrier == "shipping_hbi"].p_nom_opt.div(1e6).sum().round(1)

In [ ]:
# Top shipping routes
n.links[n.links.carrier == "shipping_hbi"].p_nom_opt.div(1e6).sort_values(ascending=False).round(1).head(20)

In [ ]:
# Top shipping routes: costs
n.links[n.links.carrier == "shipping_hbi"].marginal_cost

#p_nom_opt.div(1e6).sort_values(ascending=False).round(1).head(20)